# Chapter 6: Linear Transformations

**Book:** *Linear Algebra with Applications in Machine Learning: From Intuitive Understanding to Python Coding*

---

Linear transformations are the mechanism by which matrices act on vectors: rotating, projecting, scaling, and reshaping spaces. Every matrix multiplication $T(\mathbf{x}) = A\mathbf{x}$ is a linear transformation, and understanding these maps is essential for grasping how ML models process data. This notebook covers:

1. **Definition and Basics** — linearity properties, kernel, image, Rank-Nullity Theorem
2. **Injectivity, Surjectivity, Bijectivity** — one-to-one, onto, invertible maps
3. **Function and Matrix Inverses** — computing $A^{-1}$, properties, elementary matrices
4. **Common Transformations** — rotation, projection, reflection, shear (with visualizations)
5. **Transformations in ML** — regression, PCA, neural network layers

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import sympy as sp

plt.rcParams['figure.figsize'] = (8, 5)
plt.rcParams['figure.dpi'] = 100
plt.rcParams['font.size'] = 12
np.set_printoptions(precision=4, suppress=True)

print("All imports successful.")

## 6.1 Linear Transformations: Definition and Basics

A **linear transformation** $T: \mathbb{R}^n \to \mathbb{R}^m$ satisfies two properties for all $\mathbf{u}, \mathbf{v}$ and scalars $c$:

$$T(\mathbf{u} + \mathbf{v}) = T(\mathbf{u}) + T(\mathbf{v}) \quad \text{(additivity)}$$
$$T(c\mathbf{u}) = cT(\mathbf{u}) \quad \text{(homogeneity)}$$

Every such $T$ can be represented by a matrix $A \in \mathbb{R}^{m \times n}$: $T(\mathbf{x}) = A\mathbf{x}$. The columns of $A$ are the images of the standard basis vectors.

In [ ]:
# Verify linearity of T(x) = Ax
A = np.array([[2, 1],
              [1, 3]])

u = np.array([1, 2])
v = np.array([3, -1])
c = 5.0

# Additivity: T(u+v) == T(u) + T(v)
lhs_add = A @ (u + v)
rhs_add = A @ u + A @ v
print(f"Additivity:   T(u+v) = {lhs_add},  T(u)+T(v) = {rhs_add},  equal: {np.allclose(lhs_add, rhs_add)}")

# Homogeneity: T(cu) == cT(u)
lhs_hom = A @ (c * u)
rhs_hom = c * (A @ u)
print(f"Homogeneity:  T(cu) = {lhs_hom},  cT(u) = {rhs_hom},  equal: {np.allclose(lhs_hom, rhs_hom)}")

# The columns of A are the images of e1, e2
print(f"\nT(e1) = A @ [1,0] = {A @ np.array([1,0])}  (column 0 of A)")
print(f"T(e2) = A @ [0,1] = {A @ np.array([0,1])}  (column 1 of A)")

### Visualizing a Linear Transformation

A linear transformation maps the unit square (spanned by basis vectors) to a parallelogram defined by the columns of $A$. Let us see how several standard shapes are deformed.

In [ ]:
def plot_transformation(A, title=''):
    """Visualize how a 2x2 matrix transforms the unit square and unit circle."""
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 5.5))
    
    # Unit square
    square = np.array([[0,0],[1,0],[1,1],[0,1],[0,0]]).T
    sq_t = A @ square
    
    # Unit circle
    theta = np.linspace(0, 2*np.pi, 100)
    circle = np.array([np.cos(theta), np.sin(theta)])
    circ_t = A @ circle
    
    for ax, orig, trans, shape_name in [(ax1, square, sq_t, 'Square'),
                                         (ax2, circle, circ_t, 'Circle')]:
        ax.plot(orig[0], orig[1], 'b-', linewidth=2, label=f'Original {shape_name}')
        ax.fill(orig[0], orig[1], alpha=0.1, color='blue')
        ax.plot(trans[0], trans[1], 'r-', linewidth=2, label=f'Transformed')
        ax.fill(trans[0], trans[1], alpha=0.1, color='red')
        
        # Basis vectors and their images
        ax.quiver(0, 0, 1, 0, angles='xy', scale_units='xy', scale=1,
                  color='blue', linewidth=1.5, alpha=0.6)
        ax.quiver(0, 0, 0, 1, angles='xy', scale_units='xy', scale=1,
                  color='blue', linewidth=1.5, alpha=0.6)
        ax.quiver(0, 0, A[0,0], A[1,0], angles='xy', scale_units='xy', scale=1,
                  color='red', linewidth=1.5, alpha=0.8)
        ax.quiver(0, 0, A[0,1], A[1,1], angles='xy', scale_units='xy', scale=1,
                  color='darkred', linewidth=1.5, alpha=0.8)
        
        lim = max(3, np.max(np.abs(trans)) + 0.5)
        ax.set_xlim(-lim, lim)
        ax.set_ylim(-lim, lim)
        ax.set_aspect('equal')
        ax.legend(fontsize=9)
        ax.grid(True, alpha=0.3)
        ax.axhline(0, color='gray', linewidth=0.5)
        ax.axvline(0, color='gray', linewidth=0.5)
    
    plt.suptitle(f'{title}:  A = {A.tolist()}', fontsize=13, fontweight='bold')
    plt.tight_layout()
    plt.show()

plot_transformation(np.array([[2, 1], [0, 1.5]]), 'Stretch + Shear')

### Kernel and Image

- **Kernel** (null space): $\ker(T) = \{\mathbf{x} \mid A\mathbf{x} = \mathbf{0}\}$ — vectors mapped to zero
- **Image** (column space): $\text{im}(T) = \{A\mathbf{x} \mid \mathbf{x} \in \mathbb{R}^n\}$ — all reachable outputs
- **Rank-Nullity Theorem**: $\dim(\ker(T)) + \dim(\text{im}(T)) = n$

In [ ]:
# Example: rank-deficient matrix
A = sp.Matrix([[1, 2],
               [3, 6]])

print("A =")
sp.pprint(A)

# Kernel
kernel = A.nullspace()
print(f"\nKernel (null space):")
for v in kernel:
    print(f"  span of {v.T}")

# Image (column space)
col_space = A.columnspace()
print(f"\nImage (column space):")
for v in col_space:
    print(f"  span of {v.T}")

# Rank-Nullity
rank = A.rank()
nullity = len(kernel)
n = A.cols
print(f"\nRank = {rank}, Nullity = {nullity}, n = {n}")
print(f"Rank + Nullity = {rank + nullity} = n? {rank + nullity == n}")

In [ ]:
# Visualize kernel and image
A_np = np.array([[1, 2], [3, 6]], dtype=float)

fig, ax = plt.subplots(figsize=(7, 7))
t = np.linspace(-2, 2, 100)

# Kernel: x1 + 2*x2 = 0  =>  x1 = -2*x2
ax.plot(-2*t, t, 'b-', linewidth=2.5, label=r'Kernel: $x_1 = -2x_2$')

# Image: span of [1, 3]
ax.plot(t, 3*t, 'r--', linewidth=2.5, label='Image: span([1, 3])')

ax.set_xlim(-4, 4)
ax.set_ylim(-4, 4)
ax.set_aspect('equal')
ax.set_xlabel(r'$x_1$')
ax.set_ylabel(r'$x_2$')
ax.set_title('Kernel and Image of A = [[1,2],[3,6]]', fontsize=13)
ax.legend(fontsize=11)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

### Injectivity, Surjectivity, Bijectivity

| Property | Condition | Meaning |
|----------|-----------|--------|
| Injective (1-to-1) | $\ker(T) = \{\mathbf{0}\}$, rank = $n$ | Distinct inputs give distinct outputs |
| Surjective (onto) | $\text{im}(T) = \mathbb{R}^m$, rank = $m$ | Every output is reachable |
| Bijective | Both, requires $m = n$, full rank | Invertible |

In [ ]:
def classify_transformation(A, name="T"):
    """Classify a linear transformation by injectivity/surjectivity."""
    m, n = A.shape
    rank = np.linalg.matrix_rank(A)
    injective = (rank == n)
    surjective = (rank == m)
    bijective = injective and surjective
    
    print(f"--- {name}: A is {m}x{n}, rank = {rank} ---")
    print(f"  Injective (rank={n})?  {injective}")
    print(f"  Surjective (rank={m})? {surjective}")
    print(f"  Bijective?             {bijective}")
    if bijective:
        print(f"  => A is invertible")
    print()

# Rank-deficient: neither
classify_transformation(np.array([[1,2],[3,6]], dtype=float), "Rank-1 (2x2)")

# Full rank square: bijective
classify_transformation(np.array([[1,2],[3,5]], dtype=float), "Full-rank (2x2)")

# Tall matrix: injective but not surjective
classify_transformation(np.array([[1,0],[0,1],[1,1]], dtype=float), "Tall (3x2)")

# Wide matrix: surjective but not injective
classify_transformation(np.array([[1,0,1],[0,1,1]], dtype=float), "Wide (2x3)")

## 6.2 Function and Matrix Inverses

A function $f$ has an inverse $f^{-1}$ if and only if it is bijective. For matrices, $A^{-1}$ exists when $A$ is square with $\det(A) \neq 0$, satisfying $A A^{-1} = A^{-1} A = I$.

### Function Inverse Example

In [ ]:
# f(x) = 3x + 2,  f_inv(x) = (x - 2) / 3
f = lambda x: 3*x + 2
f_inv = lambda x: (x - 2) / 3

x_test = 5
print(f"f(f_inv({x_test})) = {f(f_inv(x_test))}  (should be {x_test})")
print(f"f_inv(f({x_test})) = {f_inv(f(x_test))}  (should be {x_test})")

# Plot
fig, ax = plt.subplots(figsize=(8, 6))
x_range = np.linspace(-3, 5, 200)
ax.plot(x_range, f(x_range), 'b-', linewidth=2, label=r'$f(x) = 3x + 2$')
ax.plot(x_range, f_inv(x_range), 'r--', linewidth=2, label=r'$f^{-1}(x) = \frac{x-2}{3}$')
ax.plot(x_range, x_range, 'k:', linewidth=1, alpha=0.5, label=r'$y = x$')
ax.set_xlabel('x')
ax.set_ylabel('y')
ax.set_title('A Function and Its Inverse (Reflected Over y = x)', fontsize=13)
ax.legend(fontsize=11)
ax.grid(True, alpha=0.3)
ax.set_aspect('equal')
ax.set_xlim(-3, 8)
ax.set_ylim(-3, 8)
plt.tight_layout()
plt.show()

### 2x2 Matrix Inverse

For $A = \begin{bmatrix} a & b \\ c & d \end{bmatrix}$ with $\det(A) = ad - bc \neq 0$:

$$A^{-1} = \frac{1}{ad - bc} \begin{bmatrix} d & -b \\ -c & a \end{bmatrix}$$

In [ ]:
A = np.array([[1, 2],
              [3, 5]])

det_A = np.linalg.det(A)
A_inv = np.linalg.inv(A)

print(f"A =\n{A}")
print(f"\ndet(A) = {det_A:.4f}")
print(f"\nA^(-1) =\n{A_inv}")
print(f"\nVerification: A @ A^(-1) =\n{(A @ A_inv).round(10)}")

In [ ]:
# Manual 2x2 inverse formula
def inv_2x2(M):
    a, b = M[0, 0], M[0, 1]
    c, d = M[1, 0], M[1, 1]
    det = a*d - b*c
    if abs(det) < 1e-12:
        raise ValueError("Matrix is singular")
    return (1/det) * np.array([[d, -b], [-c, a]])

A_inv_manual = inv_2x2(A)
print(f"Manual formula: A^(-1) =\n{A_inv_manual}")
print(f"Matches np.linalg.inv: {np.allclose(A_inv, A_inv_manual)}")

### Solving $A\mathbf{x} = \mathbf{b}$ via Inverse

If $A$ is invertible, $\mathbf{x} = A^{-1}\mathbf{b}$.

In [ ]:
# Solve: x1 + 2*x2 = 4,  3*x1 + 5*x2 = 7
A = np.array([[1, 2], [3, 5]])
b = np.array([4, 7])

x_inv = np.linalg.inv(A) @ b
x_solve = np.linalg.solve(A, b)

print(f"Via inverse:  x = {x_inv}")
print(f"Via solve:    x = {x_solve}")
print(f"Verify: A @ x = {A @ x_inv}  (should be {b})")

# Visualize
fig, ax = plt.subplots(figsize=(7, 6))
x1_r = np.linspace(-8, 4, 200)
ax.plot(x1_r, (4 - x1_r)/2, 'b-', linewidth=2, label=r'$x_1 + 2x_2 = 4$')
ax.plot(x1_r, (7 - 3*x1_r)/5, 'r--', linewidth=2, label=r'$3x_1 + 5x_2 = 7$')
ax.plot(x_inv[0], x_inv[1], 'ko', markersize=10)
ax.annotate(f'({x_inv[0]:.0f}, {x_inv[1]:.0f})', xy=x_inv, xytext=(-4, 4),
            fontsize=12, arrowprops=dict(arrowstyle='->', color='black'))
ax.set_xlabel(r'$x_1$')
ax.set_ylabel(r'$x_2$')
ax.set_title(r'Solving $A\mathbf{x} = \mathbf{b}$ via $A^{-1}$', fontsize=13)
ax.legend(fontsize=11)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

### Inverse via Gauss-Jordan Elimination

Transform $[A \mid I] \to [I \mid A^{-1}]$ using row operations. SymPy handles this exactly.

In [ ]:
# 3x3 inverse via Gauss-Jordan (SymPy)
A = sp.Matrix([[1, 0, 4],
               [0, 2, 3],
               [0, 0, 1]])

print("A =")
sp.pprint(A)

A_inv = A.inv()
print("\nA^(-1) =")
sp.pprint(A_inv)

print("\nVerification: A * A^(-1) =")
sp.pprint(A * A_inv)

### Singular (Non-Invertible) Matrices

Not every square matrix is invertible. If $\det(A) = 0$, the matrix is singular.

In [ ]:
A_singular = np.array([[1, 2], [0, 0]])

print(f"A =\n{A_singular}")
print(f"det(A) = {np.linalg.det(A_singular):.4f}")
print(f"rank(A) = {np.linalg.matrix_rank(A_singular)}")

try:
    np.linalg.inv(A_singular)
    print("Inverse exists.")
except np.linalg.LinAlgError:
    print("LinAlgError: Matrix is singular, no inverse exists.")

# Why: second row is all zeros => rank < n => not full rank
# For ANY choice of A^{-1}, row 2 of A @ A^{-1} will be [0, 0] != [0, 1]

### Properties of Matrix Inverses

In [ ]:
A = np.array([[1, 0], [0, 2]])
B = np.array([[2, 0], [0, 1]])

# Property 1: (AB)^{-1} = B^{-1} A^{-1}
AB_inv = np.linalg.inv(A @ B)
B_inv_A_inv = np.linalg.inv(B) @ np.linalg.inv(A)
print(f"(AB)^(-1) =\n{AB_inv}")
print(f"B^(-1) A^(-1) =\n{B_inv_A_inv}")
print(f"Equal: {np.allclose(AB_inv, B_inv_A_inv)}\n")

# Property 2: (A^T)^{-1} = (A^{-1})^T
AT_inv = np.linalg.inv(A.T)
A_inv_T = np.linalg.inv(A).T
print(f"(A^T)^(-1) =\n{AT_inv}")
print(f"(A^(-1))^T =\n{A_inv_T}")
print(f"Equal: {np.allclose(AT_inv, A_inv_T)}")

### Elementary Matrices

Elementary matrices encode single row operations (swap, scale, add). Every invertible matrix is a product of elementary matrices, and each elementary matrix is itself invertible.

In [ ]:
# Three types of elementary matrices for 3x3
I3 = np.eye(3)

# 1. Swap rows 0 and 1
E_swap = np.array([[0, 1, 0], [1, 0, 0], [0, 0, 1]], dtype=float)
print(f"Swap (rows 0,1):\n{E_swap}")
print(f"Self-inverse: E @ E = I? {np.allclose(E_swap @ E_swap, I3)}\n")

# 2. Scale row 1 by -4
E_scale = np.diag([1, -4, 1]).astype(float)
E_scale_inv = np.diag([1, -1/4, 1])
print(f"Scale row 1 by -4:\n{E_scale}")
print(f"Its inverse (scale by -1/4):\n{E_scale_inv}")
print(f"E @ E_inv = I? {np.allclose(E_scale @ E_scale_inv, I3)}\n")

# 3. Add 2 * row 0 to row 2
E_add = np.array([[1, 0, 0], [0, 1, 0], [2, 0, 1]], dtype=float)
E_add_inv = np.array([[1, 0, 0], [0, 1, 0], [-2, 0, 1]], dtype=float)
print(f"Add 2*row0 to row2:\n{E_add}")
print(f"Its inverse (add -2*row0):\n{E_add_inv}")
print(f"E @ E_inv = I? {np.allclose(E_add @ E_add_inv, I3)}")

In [ ]:
# Applying an elementary matrix = performing a row operation
A = np.array([[1, 4], [2, 5], [3, 6]])

print(f"Original A:\n{A}\n")
print(f"After swapping rows 0 and 1 (E_swap @ A):\n{E_swap @ A}")

## 6.3 Common Transformations

### 6.3.1 Rotation

A rotation by angle $\theta$ in $\mathbb{R}^2$ preserves lengths and angles:

$$R(\theta) = \begin{bmatrix} \cos\theta & -\sin\theta \\ \sin\theta & \cos\theta \end{bmatrix}$$

Properties: $\det(R) = 1$, $R^{-1} = R^T = R(-\theta)$.

In [ ]:
def rotation_matrix(theta_deg):
    theta = np.radians(theta_deg)
    return np.array([[np.cos(theta), -np.sin(theta)],
                     [np.sin(theta),  np.cos(theta)]])

# Rotate [1, 0] by various angles
v = np.array([1, 0])
angles = [30, 45, 60, 90, 135, 180]

fig, ax = plt.subplots(figsize=(7, 7))
circle = plt.Circle((0,0), 1, fill=False, color='gray', linestyle='--', linewidth=1)
ax.add_patch(circle)

ax.quiver(0, 0, v[0], v[1], angles='xy', scale_units='xy', scale=1,
          color='blue', linewidth=2.5, label='Original')

colors = plt.cm.Reds(np.linspace(0.3, 0.9, len(angles)))
for ang, col in zip(angles, colors):
    R = rotation_matrix(ang)
    v_rot = R @ v
    ax.quiver(0, 0, v_rot[0], v_rot[1], angles='xy', scale_units='xy', scale=1,
              color=col, linewidth=2, label=f'{ang}deg')

ax.set_xlim(-1.5, 1.5)
ax.set_ylim(-1.3, 1.5)
ax.set_aspect('equal')
ax.set_title('Rotation Preserves Length (Tip Stays on Unit Circle)', fontsize=13)
ax.legend(fontsize=9, ncol=2)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
# Verify rotation properties
R45 = rotation_matrix(45)
print(f"R(45 deg) =\n{R45.round(4)}")
print(f"det(R) = {np.linalg.det(R45):.6f}  (should be 1)")
print(f"R^T @ R = I? {np.allclose(R45.T @ R45, np.eye(2))}  (orthogonal)")
print(f"R^(-1) = R^T? {np.allclose(np.linalg.inv(R45), R45.T)}")

# Rotate a shape
plot_transformation(R45, 'Rotation (45 deg)')

### 6.3.2 Projection

Projection onto $\text{span}(\mathbf{u})$:

$$P = \frac{\mathbf{u}\mathbf{u}^T}{\mathbf{u}^T\mathbf{u}}$$

Properties: $P^2 = P$ (idempotent), $P^T = P$ (symmetric), rank = 1.

In [ ]:
# Projection onto u = [1, 1]
u = np.array([1, 1])
P = np.outer(u, u) / np.dot(u, u)

v = np.array([2, 1])
v_proj = P @ v

print(f"Projection matrix P =\n{P}")
print(f"\nv = {v}")
print(f"P @ v = {v_proj}")
print(f"\nP^2 = P (idempotent)? {np.allclose(P @ P, P)}")
print(f"P^T = P (symmetric)?  {np.allclose(P, P.T)}")

# Residual is orthogonal to u
residual = v - v_proj
print(f"\nResidual: v - Pv = {residual}")
print(f"Residual . u = {np.dot(residual, u):.6f}  (should be 0: orthogonal)")

In [ ]:
# Visualize projection
fig, ax = plt.subplots(figsize=(7, 7))

# Projection line
t = np.linspace(-1, 3, 100)
u_norm = u / np.linalg.norm(u)
ax.plot(t * u_norm[0], t * u_norm[1], 'k--', linewidth=1.5, alpha=0.5,
        label=r'Subspace: $x_1 = x_2$')

# Vectors
ax.quiver(0, 0, v[0], v[1], angles='xy', scale_units='xy', scale=1,
          color='blue', linewidth=2.5, label=f'v = {v}')
ax.quiver(0, 0, v_proj[0], v_proj[1], angles='xy', scale_units='xy', scale=1,
          color='red', linewidth=2.5, label=f'Pv = {v_proj}')

# Residual (perpendicular drop)
ax.plot([v[0], v_proj[0]], [v[1], v_proj[1]], 'g:', linewidth=2, label='Residual')

# Right-angle marker
s = 0.15
perp = residual / np.linalg.norm(residual) * s
along = u_norm * s
corner = np.array([v_proj + perp, v_proj + perp + along, v_proj + along])
ax.plot(corner[:, 0], corner[:, 1], 'g-', linewidth=1.5)

ax.set_xlim(-0.5, 3)
ax.set_ylim(-0.5, 2.5)
ax.set_aspect('equal')
ax.set_xlabel(r'$x_1$')
ax.set_ylabel(r'$x_2$')
ax.set_title('Orthogonal Projection onto span([1,1])', fontsize=13)
ax.legend(fontsize=10)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

### 6.3.3 Reflection

Reflection over a line through the origin with direction $\mathbf{u}$ uses the Householder formula:

$$H = I - \frac{2\mathbf{n}\mathbf{n}^T}{\mathbf{n}^T\mathbf{n}}$$

where $\mathbf{n}$ is the normal to the reflection line. For reflection over the x-axis, $H = \begin{bmatrix} 1 & 0 \\ 0 & -1 \end{bmatrix}$.

Alternatively, reflection *about* the line $\text{span}(\mathbf{u})$ is $2P - I$ where $P$ is the projection.

In [ ]:
# Reflection over x-axis (negate y-component)
H_x = np.array([[-1, 0],
                [ 0, 1]])

v = np.array([1, 1])
v_ref = H_x @ v

print(f"Reflection matrix (over y-axis): {H_x.tolist()}")
print(f"v = {v}, reflected = {v_ref}")
print(f"det(H) = {np.linalg.det(H_x):.0f}  (should be -1 for reflections)")
print(f"H^2 = I? {np.allclose(H_x @ H_x, np.eye(2))}  (self-inverse)")

plot_transformation(H_x, 'Reflection Over y-axis')

### 6.3.4 Shear

A horizontal shear shifts $x_1$ by an amount proportional to $x_2$:

$$S = \begin{bmatrix} 1 & k \\ 0 & 1 \end{bmatrix}$$

Shears preserve area ($\det(S) = 1$) and are always invertible: $S^{-1} = \begin{bmatrix} 1 & -k \\ 0 & 1 \end{bmatrix}$.

In [ ]:
k = 2
S = np.array([[1, k], [0, 1]])
S_inv = np.array([[1, -k], [0, 1]])

v = np.array([1, 1])
v_shear = S @ v

print(f"Shear matrix (k={k}): {S.tolist()}")
print(f"v = {v}, sheared = {v_shear}")
print(f"det(S) = {np.linalg.det(S):.0f}  (area preserved)")
print(f"S @ S_inv = I? {np.allclose(S @ S_inv, np.eye(2))}")

plot_transformation(S, f'Horizontal Shear (k={k})')

### 6.3.5 All Four Transformations Compared

In [ ]:
transforms = [
    ('Rotation (45 deg)', rotation_matrix(45)),
    ('Projection onto [1,1]', np.outer([1,1],[1,1]) / 2),
    ('Reflection (y-axis)', np.array([[-1,0],[0,1]])),
    ('Shear (k=1.5)', np.array([[1, 1.5],[0, 1]])),
]

fig, axes = plt.subplots(1, 4, figsize=(20, 5))
theta = np.linspace(0, 2*np.pi, 100)
circle = np.array([np.cos(theta), np.sin(theta)])

for ax, (name, M) in zip(axes, transforms):
    transformed = M @ circle
    ax.plot(circle[0], circle[1], 'b-', linewidth=1.5, alpha=0.5)
    ax.plot(transformed[0], transformed[1], 'r-', linewidth=2)
    ax.set_title(name, fontsize=11)
    ax.set_xlim(-2.5, 2.5)
    ax.set_ylim(-2.5, 2.5)
    ax.set_aspect('equal')
    ax.grid(True, alpha=0.3)
    ax.axhline(0, color='gray', linewidth=0.5)
    ax.axvline(0, color='gray', linewidth=0.5)

plt.suptitle('Four Common Linear Transformations (Unit Circle)', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

### Transformation Properties Summary

| Transformation | det | Invertible | Preserves Lengths | Preserves Area |
|---------------|-----|------------|-------------------|----------------|
| Rotation | 1 | Yes | Yes | Yes |
| Projection | 0 | No | No | No |
| Reflection | -1 | Yes | Yes | Yes |
| Shear | 1 | Yes | No | Yes |

In [ ]:
# Verify the properties table
print(f"{'Name':25s} {'det':>6s} {'Invertible':>12s} {'Orthogonal':>12s}")
print("-" * 58)
for name, M in transforms:
    d = np.linalg.det(M)
    inv = abs(d) > 1e-10
    orth = np.allclose(M.T @ M, np.eye(2))
    print(f"{name:25s} {d:6.2f} {str(inv):>12s} {str(orth):>12s}")

## 6.4 Transformations in Machine Learning

### 6.4.1 Linear Regression via Normal Equations

The closed-form solution $\mathbf{w} = (X^T X)^{-1} X^T \mathbf{y}$ uses the matrix inverse to find optimal weights.

In [ ]:
X = np.array([[1, 2], [3, 4], [5, 6]])
y = np.array([1, 2, 3])

w = np.linalg.inv(X.T @ X) @ X.T @ y
print(f"Feature matrix X:\n{X}")
print(f"Target y: {y}")
print(f"Weights w = (X^T X)^(-1) X^T y = {w}")
print(f"Predictions: X @ w = {X @ w}")

### 6.4.2 Neural Network Layer as a Transformation

Each layer computes $\mathbf{h} = \sigma(W\mathbf{x} + \mathbf{b})$, where $W\mathbf{x}$ is the linear part and $\sigma$ adds non-linearity. Without $\sigma$, stacking layers collapses to a single linear map.

In [ ]:
np.random.seed(7)

# Two layers: 3 -> 4 -> 2
W1 = np.random.randn(4, 3) * 0.5
b1 = np.zeros(4)
W2 = np.random.randn(2, 4) * 0.5
b2 = np.zeros(2)

x = np.array([1.0, -0.5, 2.0])
relu = lambda z: np.maximum(0, z)

# With activation (non-linear)
h1 = relu(W1 @ x + b1)
out_nonlinear = W2 @ h1 + b2

# Without activation (collapses to single linear map)
W_combined = W2 @ W1
out_linear = W_combined @ x + W2 @ b1 + b2

print(f"Input: {x}")
print(f"With ReLU:    {out_nonlinear.round(4)}")
print(f"Without ReLU: {out_linear.round(4)}")
print(f"\nThese differ because ReLU breaks linearity.")
print(f"Combined W2 @ W1 (2x3):\n{W_combined.round(4)}")
print(f"Without activation, two layers = one linear transformation.")

### 6.4.3 Composing Transformations

In both ML and graphics, we chain transformations by multiplying matrices. The order matters: $(AB)\mathbf{x} = A(B\mathbf{x})$ applies $B$ first.

In [ ]:
# Scale by 2, then rotate 45 degrees
Scale = np.array([[2, 0], [0, 2]])
Rot = rotation_matrix(45)

# Composition: rotate THEN scale
C1 = Scale @ Rot
# Composition: scale THEN rotate
C2 = Rot @ Scale

print(f"Scale then Rotate:\n{C1.round(4)}")
print(f"\nRotate then Scale:\n{C2.round(4)}")
print(f"\nSame result? {np.allclose(C1, C2)}")
print("(In this case yes, because uniform scaling commutes with rotation.)")

# Non-commutative example: shear then rotate vs rotate then shear
Shear = np.array([[1, 1], [0, 1]])
print(f"\nShear then Rotate:\n{(Rot @ Shear).round(4)}")
print(f"Rotate then Shear:\n{(Shear @ Rot).round(4)}")
print(f"Same? {np.allclose(Rot @ Shear, Shear @ Rot)}  (order matters!)")

## 6.5 Exercises

Selected exercises from the chapter.

**Exercise 3:** Compute the kernel of $T(\mathbf{x}) = \begin{bmatrix} 1 & 2 \\ 2 & 4 \end{bmatrix}\mathbf{x}$.

In [ ]:
# Exercise 3: Your code here


**Exercise 6:** Find the inverse of $\begin{bmatrix} 4 & 7 \\ 2 & 6 \end{bmatrix}$ using the 2x2 formula.

In [ ]:
# Exercise 6: Your code here


**Exercise 11:** Compute the image of $\mathbf{v} = [1, 1]$ under a 45 degree rotation.

In [ ]:
# Exercise 11: Your code here


**Exercise 12:** Project $\mathbf{v} = [3, 2]$ onto the line spanned by $[1, 1]$.

In [ ]:
# Exercise 12: Your code here


**Exercise 17:** For $A = \begin{bmatrix} 1 & 2 \\ 3 & 4 \end{bmatrix}$, $B = \begin{bmatrix} 2 & 1 \\ 0 & 1 \end{bmatrix}$, verify $(AB)^{-1} = B^{-1}A^{-1}$.

In [ ]:
# Exercise 17: Your code here


---

## Exercise Solutions

In [ ]:
# --- Solution: Exercise 3 ---
A = sp.Matrix([[1, 2], [2, 4]])
print("Kernel of A:")
for v in A.nullspace():
    print(f"  span({v.T})")
print(f"Nullity = {len(A.nullspace())}")

In [ ]:
# --- Solution: Exercise 6 ---
M = np.array([[4, 7], [2, 6]])
det_M = 4*6 - 7*2
M_inv = (1/det_M) * np.array([[6, -7], [-2, 4]])
print(f"det = {det_M}")
print(f"Inverse =\n{M_inv}")
print(f"Verify: M @ M_inv =\n{(M @ M_inv).round(10)}")

In [ ]:
# --- Solution: Exercise 11 ---
R45 = rotation_matrix(45)
v = np.array([1, 1])
v_rot = R45 @ v
print(f"v = {v}")
print(f"R(45 deg) @ v = {v_rot.round(4)}")
print(f"||v|| = {np.linalg.norm(v):.4f}, ||Rv|| = {np.linalg.norm(v_rot):.4f}  (preserved)")

In [ ]:
# --- Solution: Exercise 12 ---
u = np.array([1, 1])
P = np.outer(u, u) / np.dot(u, u)
v = np.array([3, 2])
proj = P @ v
print(f"Projection of {v} onto span([1,1]) = {proj}")

In [ ]:
# --- Solution: Exercise 17 ---
A = np.array([[1, 2], [3, 4]])
B = np.array([[2, 1], [0, 1]])

AB_inv = np.linalg.inv(A @ B)
B_inv_A_inv = np.linalg.inv(B) @ np.linalg.inv(A)

print(f"(AB)^(-1) =\n{AB_inv.round(4)}")
print(f"B^(-1) A^(-1) =\n{B_inv_A_inv.round(4)}")
print(f"Equal: {np.allclose(AB_inv, B_inv_A_inv)}")